# Keem Render — Kaggle GPU worker

Use a Kaggle Notebook with **GPU enabled** and **Internet enabled**. Before running, add two Kaggle Secrets under Add-ons → Secrets:
- `HF_TOKEN`: Hugging Face token after accepting the SDXL Base 1.0 model licence.
- `KEEM_RENDER_WORKER_TOKEN`: a long random secret. Keep it private; later it must match Vercel's `COMFYUI_API_KEY`.

This notebook starts ComfyUI on localhost, a token-protected gateway, and a temporary Cloudflare Quick Tunnel. Do not connect it to the public app until the app has login protection and rate limiting. Free Kaggle GPU access is limited and can queue or stop.

In [ ]:
import os, subprocess, sys, time, requests
from pathlib import Path

WORK = Path('/kaggle/working')
COMFY = WORK / 'ComfyUI'
REPO = WORK / 'keem-render'
assert Path('/kaggle').exists(), 'Run this notebook on Kaggle.'

if not COMFY.exists():
    subprocess.run(['git','clone','https://github.com/comfyanonymous/ComfyUI.git',str(COMFY)], check=True)
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/thayoungwizard/keem-render.git',str(REPO)], check=True)

subprocess.run([sys.executable,'-m','pip','install','-r',str(COMFY/'requirements.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','fastapi','uvicorn','httpx','huggingface_hub'],check=True)
print('ComfyUI and gateway dependencies installed.')

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download

secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
worker_token = secrets.get_secret('KEEM_RENDER_WORKER_TOKEN')
if not hf_token or not worker_token:
    raise RuntimeError('Add HF_TOKEN and KEEM_RENDER_WORKER_TOKEN in Kaggle Secrets first.')

checkpoint = COMFY / 'models' / 'checkpoints' / 'sd_xl_base_1.0.safetensors'
checkpoint.parent.mkdir(parents=True, exist_ok=True)
if not checkpoint.exists():
    hf_hub_download(repo_id='stabilityai/stable-diffusion-xl-base-1.0', filename='sd_xl_base_1.0.safetensors', token=hf_token, local_dir=str(checkpoint.parent))
print('SDXL checkpoint is ready.')

In [ ]:
comfy_log = open('/kaggle/working/comfyui.log','w')
comfy_proc = subprocess.Popen([sys.executable,'main.py','--listen','127.0.0.1','--port','8188','--disable-auto-launch'],cwd=COMFY,stdout=comfy_log,stderr=subprocess.STDOUT)
ready = False
for _ in range(180):
    if comfy_proc.poll() is not None:
        raise RuntimeError('ComfyUI exited. Inspect /kaggle/working/comfyui.log')
    try:
        response = requests.get('http://127.0.0.1:8188/system_stats',timeout=3)
        if response.ok:
            ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(5)
if not ready:
    raise TimeoutError('ComfyUI did not become ready. Inspect /kaggle/working/comfyui.log')
print('ComfyUI is running locally on port 8188.')

In [ ]:
gateway_env = os.environ.copy()
gateway_env['KEEM_RENDER_WORKER_TOKEN'] = worker_token
gateway_env['COMFYUI_LOCAL_URL'] = 'http://127.0.0.1:8188'
gateway_log = open('/kaggle/working/gateway.log','w')
gateway_proc = subprocess.Popen([sys.executable,'-m','uvicorn','gateway:app','--host','127.0.0.1','--port','8189'],cwd=REPO/'gpu-worker',env=gateway_env,stdout=gateway_log,stderr=subprocess.STDOUT)
for _ in range(30):
    if gateway_proc.poll() is not None:
        raise RuntimeError('Gateway exited. Inspect /kaggle/working/gateway.log')
    try:
        if requests.get('http://127.0.0.1:8189/health',timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise TimeoutError('Gateway did not become ready.')
test = requests.get('http://127.0.0.1:8189/queue',headers={'Authorization':f'Bearer {worker_token}'},timeout=10)
if not test.ok:
    raise RuntimeError(f'Authenticated gateway test failed: HTTP {test.status_code}')
print('Protected gateway is ready. The worker token was not printed.')

In [ ]:
cloudflared = '/usr/local/bin/cloudflared'
if not Path(cloudflared).exists():
    subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O',cloudflared],check=True)
    os.chmod(cloudflared,0o755)
tunnel_log = open('/kaggle/working/cloudflared.log','w')
tunnel_proc = subprocess.Popen([cloudflared,'tunnel','--url','http://127.0.0.1:8189','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
import re
tunnel_url = None
for _ in range(90):
    if tunnel_proc.poll() is not None:
        raise RuntimeError('Cloudflare tunnel exited. Inspect /kaggle/working/cloudflared.log')
    try:
        log_text = Path('/kaggle/working/cloudflared.log').read_text(errors='ignore')
        match = re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com',log_text)
        if match:
            tunnel_url = match.group(0)
            break
    except FileNotFoundError:
        pass
    time.sleep(2)
if not tunnel_url:
    raise TimeoutError('No Quick Tunnel URL appeared. Inspect cloudflared.log')
print('Temporary GPU gateway URL:',tunnel_url)
print('Keep this notebook session running. The URL changes when the tunnel restarts.')
print('Do not connect this URL to Vercel until Keem Render access protection is enabled.')

## Session notes
- Keep this notebook session active while rendering.
- Stop the session when finished to release your Kaggle GPU quota.
- Never print or share `worker_token`.
- Never expose ComfyUI port 8188 directly; only the gateway on 8189 is tunneled.
- This is a temporary testing worker, not a persistent service.